# 🎵 Salama Insurance — Claims Call Analytics Audio Generator

**Purpose:** Generate synthetic call center audio recordings from actual claims data for:
- 🗣️ **Speech-to-Text Analytics** — Test transcription pipelines with realistic insurance call content
- 📊 **Sentiment Analysis** — Train models on calls with known claim outcomes (approved, rejected, etc.)
- ✅ **Compliance Monitoring** — Validate agent scripts and regulatory adherence
- 🎯 **Agent Training** — Create scenario-based training materials from real claim patterns

**Data Source:** `salama_insurance.salama_silver.fact_claim`  
**Output:** MP3 audio files + JSON metadata in `/tmp/call_audio_files/`  
**Generated:** Dynamically from live claims data

In [0]:
%pip install gTTS pydub --quiet

In [0]:
import os
import json
import random
import io
import base64
import tempfile
from datetime import datetime, timedelta
from pathlib import Path

from gtts import gTTS

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
OUTPUT_DIR = "/tmp/call_audio_files"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"✅ Output directory ready: {OUTPUT_DIR}")
print(f"📅 Generation timestamp: {TIMESTAMP}")

In [0]:
%sql
SELECT
    CLAIM_ID,
    CLAIM_NUMBER,
    POLICY_ID,
    BUSINESS_LINE,
    CLAIM_TYPE,
    CLAIM_STATUS,
    ROUND(CLAIMED_AMOUNT, 2)   AS CLAIMED_AMOUNT,
    ROUND(APPROVED_AMOUNT, 2)  AS APPROVED_AMOUNT,
    ROUND(PAID_AMOUNT, 2)      AS PAID_AMOUNT,
    RISK_RATING,
    CAST(DAYS_TO_SETTLE AS INT) AS DAYS_TO_SETTLE,
    FINDINGS,
    CL_DATE
FROM salama_insurance.salama_silver.fact_claim
WHERE CLAIM_ID IS NOT NULL
ORDER BY RAND()
LIMIT 10

In [0]:
def generate_call_transcript(claim):
    """Generate a realistic call transcript from a claim record.
    
    Returns:
        transcript (list[dict]): List of {'speaker': ..., 'text': ...} turns
        full_text (str): Combined text for TTS
        metadata (dict): Call metadata
    """
    # Safely extract fields with defaults
    claim_id     = str(claim.get('CLAIM_ID', 'UNKNOWN'))
    claim_num    = str(claim.get('CLAIM_NUMBER', claim_id))
    policy_id    = str(claim.get('POLICY_ID', 'N/A'))
    biz_line     = str(claim.get('BUSINESS_LINE', 'General')).replace('_', ' ').title()
    claim_type   = str(claim.get('CLAIM_TYPE', 'General')).replace('_', ' ').title()
    status       = str(claim.get('CLAIM_STATUS', 'REPORTED')).upper()
    import math
    _claimed  = claim.get('CLAIMED_AMOUNT', 0)
    _approved = claim.get('APPROVED_AMOUNT', 0)
    _paid     = claim.get('PAID_AMOUNT', 0)
    _days     = claim.get('DAYS_TO_SETTLE', 0)
    claimed_amt  = 0.0 if (_claimed is None or (isinstance(_claimed, float) and math.isnan(_claimed))) else float(_claimed)
    approved_amt = 0.0 if (_approved is None or (isinstance(_approved, float) and math.isnan(_approved))) else float(_approved)
    paid_amt     = 0.0 if (_paid is None or (isinstance(_paid, float) and math.isnan(_paid))) else float(_paid)
    days_settle  = 0 if (_days is None or (isinstance(_days, float) and math.isnan(_days))) else int(_days)
    risk_rating  = str(claim.get('RISK_RATING', 'LOW') or 'LOW').title()
    findings     = str(claim.get('FINDINGS', '') or 'None')

    agent_names = ["Sarah", "Ahmed", "Fatima", "Omar", "Layla", "Khalid", "Nora", "Yusuf"]
    cust_names  = ["Mr. Al-Rashid", "Mrs. Hassan", "Mr. Malik", "Mrs. Noor",
                   "Mr. Ibrahim", "Mrs. Saleh", "Mr. Khan", "Mrs. Farouk"]
    agent_name = random.choice(agent_names)
    cust_name  = random.choice(cust_names)

    transcript = []

    # --- Opening ---
    transcript.append({
        "speaker": "Agent",
        "text": (f"Thank you for calling Salama Insurance claims department, "
                 f"my name is {agent_name}. How may I assist you today?")
    })

    if status == 'REPORTED':
        scenario = "New Claim Status Inquiry"
        sentiment = "anxious"
        transcript.extend([
            {"speaker": "Customer",
             "text": (f"Hello {agent_name}, this is {cust_name}. I recently filed a "
                      f"{claim_type.lower()} claim under my {biz_line.lower()} policy, "
                      f"claim number {claim_num}. I wanted to check on its status.")},
            {"speaker": "Agent",
             "text": (f"Of course, {cust_name}. Let me pull up your claim. "
                      f"I can see claim {claim_num} for your {biz_line.lower()} policy, "
                      f"policy number {policy_id}. The claimed amount is "
                      f"{claimed_amt:,.2f} dirhams.")},
            {"speaker": "Customer",
             "text": "Yes, that's correct. Has there been any progress?"},
            {"speaker": "Agent",
             "text": (f"Your claim has been received and is currently in the reported stage. "
                      f"Our assessors will begin reviewing your {claim_type.lower()} claim shortly. "
                      f"Based on similar cases, the review typically takes 7 to 14 business days.")},
            {"speaker": "Customer",
             "text": "I understand. Is there anything else I need to provide?"},
            {"speaker": "Agent",
             "text": (f"At this point, your documentation appears complete. If our team needs "
                      f"additional information, we will contact you directly. You can also track "
                      f"your claim status through our online portal using claim number {claim_num}.")},
        ])

    elif status == 'UNDER_INVESTIGATION':
        scenario = "Investigation Status Update"
        sentiment = "concerned"
        transcript.extend([
            {"speaker": "Customer",
             "text": (f"Hi, this is {cust_name}. I'm calling about claim {claim_num}. "
                      f"I received a letter saying my claim is under investigation. "
                      f"Can you tell me what that means?")},
            {"speaker": "Agent",
             "text": (f"Absolutely, {cust_name}. Let me review your file. I see your "
                      f"{claim_type.lower()} claim for {claimed_amt:,.2f} dirhams is "
                      f"currently under a routine investigation. This is a standard part "
                      f"of our process for {biz_line.lower()} claims of this nature.")},
            {"speaker": "Customer",
             "text": "How long will this investigation take? I really need this resolved."},
            {"speaker": "Agent",
             "text": (f"I completely understand your concern. The investigation has been "
                      f"ongoing for approximately {days_settle} days. Our team is working "
                      f"diligently to complete the review. The risk assessment is rated as "
                      f"{risk_rating.lower()}, and we aim to have findings within the next two weeks.")},
            {"speaker": "Customer",
             "text": "Will I be notified of the outcome?"},
            {"speaker": "Agent",
             "text": (f"Yes, absolutely. Once the investigation concludes, you will receive "
                      f"a detailed written communication with the findings and next steps. "
                      f"You may also reach us anytime at this number for updates.")},
        ])

    elif status in ('APPROVED', 'SETTLED'):
        scenario = "Approved Claim / Payment Inquiry"
        sentiment = "relieved"
        transcript.extend([
            {"speaker": "Customer",
             "text": (f"Good day, this is {cust_name}. I received notification that my "
                      f"claim {claim_num} has been {status.lower()}. I'm calling to "
                      f"confirm the payment details.")},
            {"speaker": "Agent",
             "text": (f"Congratulations, {cust_name}. Let me pull up the details. "
                      f"Your {claim_type.lower()} claim under {biz_line.lower()} policy "
                      f"{policy_id} has indeed been approved. The original claimed amount "
                      f"was {claimed_amt:,.2f} dirhams, and the approved amount is "
                      f"{approved_amt:,.2f} dirhams.")},
            {"speaker": "Customer",
             "text": (f"I see. And when can I expect to receive the payment?")},
            {"speaker": "Agent",
             "text": (f"The payment of {paid_amt:,.2f} dirhams has been processed. "
                      f"{'It should reflect in your account within 3 to 5 business days.' if paid_amt > 0 else 'Our finance team is finalizing the disbursement.'} "
                      f"The total processing time for your claim was {days_settle} days.")},
            {"speaker": "Customer",
             "text": "Thank you so much. That's a relief."},
            {"speaker": "Agent",
             "text": (f"You're most welcome. Is there anything else I can help you with "
                      f"regarding your {biz_line.lower()} policy?")},
        ])

    elif status in ('REJECTED', 'CLOSED'):
        scenario = "Claim Denial / Dispute"
        sentiment = "frustrated"
        transcript.extend([
            {"speaker": "Customer",
             "text": (f"Hello, this is {cust_name}. I just received a letter that my "
                      f"claim {claim_num} has been {status.lower()}. I'd like to "
                      f"understand why and what my options are.")},
            {"speaker": "Agent",
             "text": (f"I'm sorry to hear about this, {cust_name}. Let me review your case. "
                      f"Your {claim_type.lower()} claim for {claimed_amt:,.2f} dirhams "
                      f"under your {biz_line.lower()} policy was reviewed thoroughly.")},
            {"speaker": "Customer",
             "text": (f"I filed this claim in good faith. The claimed amount was "
                      f"{claimed_amt:,.2f} dirhams. Can you explain the reason?")},
            {"speaker": "Agent",
             "text": (f"I understand your frustration, {cust_name}. Based on our records, "
                      f"the investigation findings were: {findings.replace('_', ' ').lower()}. "
                      f"The risk assessment for this claim was rated {risk_rating.lower()}. "
                      f"{'The approved amount was ' + f'{approved_amt:,.2f} dirhams, which differs from the claimed amount.' if approved_amt > 0 else 'Unfortunately, the claim did not meet the coverage criteria.'}")},
            {"speaker": "Customer",
             "text": "Is there an appeals process I can follow?"},
            {"speaker": "Agent",
             "text": (f"Yes, absolutely. You have the right to file a formal appeal within "
                      f"30 days. I can email you the appeals form and our dispute resolution "
                      f"guidelines. You may also submit additional supporting documentation "
                      f"for review by our senior claims committee.")},
        ])

    else:
        scenario = "General Claim Inquiry"
        sentiment = "neutral"
        transcript.extend([
            {"speaker": "Customer",
             "text": (f"Hello, this is {cust_name}. I'm calling about my claim "
                      f"{claim_num} on policy {policy_id}.")},
            {"speaker": "Agent",
             "text": (f"Let me pull that up for you, {cust_name}. I see your "
                      f"{claim_type.lower()} claim under {biz_line.lower()} for "
                      f"{claimed_amt:,.2f} dirhams. The current status is {status.lower()}.")},
            {"speaker": "Customer",
             "text": "Can you provide more details on what happens next?"},
            {"speaker": "Agent",
             "text": (f"Certainly. Your claim is currently being processed by our team. "
                      f"The risk rating is {risk_rating.lower()} and we will keep you "
                      f"updated on any progress.")},
        ])

    # --- Closing ---
    transcript.extend([
        {"speaker": "Customer", "text": "That's all I needed. Thank you for your help."},
        {"speaker": "Agent",
         "text": (f"You're welcome, {cust_name}. Thank you for choosing Salama Insurance. "
                  f"If you have any further questions, please don't hesitate to call us. "
                  f"Have a wonderful day.")},
    ])

    # --- Full text for TTS ---
    full_text = " ... ".join(f"{t['speaker']}: {t['text']}" for t in transcript)
    word_count = len(full_text.split())
    duration_est = round(word_count / 150 * 60, 1)

    metadata = {
        "call_id": f"CALL-{claim_id}-{TIMESTAMP}",
        "claim_id": claim_id,
        "claim_number": claim_num,
        "policy_id": policy_id,
        "business_line": biz_line,
        "claim_type": claim_type,
        "claim_status": status,
        "scenario_type": scenario,
        "sentiment": sentiment,
        "risk_rating": risk_rating,
        "claimed_amount": claimed_amt,
        "approved_amount": approved_amt,
        "word_count": word_count,
        "estimated_duration_seconds": duration_est,
        "generated_at": datetime.now().isoformat(),
    }

    return transcript, full_text, metadata

print("✅ Transcript generator function defined")
print("   Scenarios: REPORTED, UNDER_INVESTIGATION, APPROVED/SETTLED, REJECTED/CLOSED, General")

In [0]:
def generate_call_audio(transcript_text, claim_id, output_dir):
    """Convert transcript text to MP3 audio using gTTS.
    
    Args:
        transcript_text: Full call transcript as a string
        claim_id: Claim identifier for file naming
        output_dir: Directory to save the audio file
    
    Returns:
        file_path (str): Path to the generated MP3 file, or None on failure
        file_size (int): File size in bytes
    """
    try:
        safe_id = claim_id.replace('/', '_').replace(' ', '_')
        filename = f"call_{safe_id}_{TIMESTAMP}.mp3"
        file_path = os.path.join(output_dir, filename)

        tts = gTTS(text=transcript_text, lang='en', slow=False)
        tts.save(file_path)

        file_size = os.path.getsize(file_path)
        return file_path, file_size

    except Exception as e:
        print(f"   ⚠️ Error generating audio for {claim_id}: {e}")
        return None, 0

print("✅ Audio generation function defined (gTTS → MP3)")

In [0]:
# Convert SQL results to pandas
import pandas as pd

df_claims = _sqldf.toPandas()
print(f"📎 Loaded {len(df_claims)} claims for audio generation\n")

# --- Main generation loop ---
results = []
all_transcripts = []

for idx, row in df_claims.iterrows():
    claim_id = str(row['CLAIM_ID'])
    status = str(row['CLAIM_STATUS'])
    print(f"\n🎤 [{idx+1}/{len(df_claims)}] Generating call for {claim_id} (Status: {status})")

    # Generate transcript
    transcript, full_text, metadata = generate_call_transcript(row.to_dict())
    print(f"   📝 Transcript: {metadata['word_count']} words, ~{metadata['estimated_duration_seconds']}s")
    print(f"   🎭 Scenario: {metadata['scenario_type']} | Sentiment: {metadata['sentiment']}")

    # Generate audio
    file_path, file_size = generate_call_audio(full_text, claim_id, OUTPUT_DIR)

    if file_path:
        size_kb = round(file_size / 1024, 1)
        print(f"   ✅ Audio saved: {os.path.basename(file_path)} ({size_kb} KB)")

        results.append({
            "claim_id": claim_id,
            "claim_number": metadata['claim_number'],
            "scenario": metadata['scenario_type'],
            "sentiment": metadata['sentiment'],
            "business_line": metadata['business_line'],
            "status": status,
            "words": metadata['word_count'],
            "est_duration_sec": metadata['estimated_duration_seconds'],
            "file_size_kb": size_kb,
            "file_path": file_path,
        })

        all_transcripts.append({
            "metadata": metadata,
            "transcript": transcript,
        })
    else:
        print(f"   ❌ Skipped (audio generation failed)")

print(f"\n{'='*60}")
print(f"✅ Successfully generated {len(results)} / {len(df_claims)} audio files")
print(f"{'='*60}")

# Display summary
df_results = pd.DataFrame(results)
display(df_results)

In [0]:
# Save individual JSON sidecar files + combined metadata
for entry in all_transcripts:
    meta = entry['metadata']
    safe_id = meta['claim_id'].replace('/', '_').replace(' ', '_')
    json_path = os.path.join(OUTPUT_DIR, f"call_{safe_id}_{TIMESTAMP}.json")

    sidecar = {
        "call_id": meta['call_id'],
        "claim_id": meta['claim_id'],
        "claim_number": meta['claim_number'],
        "policy_id": meta['policy_id'],
        "timestamp": meta['generated_at'],
        "duration_seconds": meta['estimated_duration_seconds'],
        "scenario_type": meta['scenario_type'],
        "sentiment": meta['sentiment'],
        "business_line": meta['business_line'],
        "claim_type": meta['claim_type'],
        "claim_status": meta['claim_status'],
        "risk_rating": meta['risk_rating'],
        "claimed_amount": meta['claimed_amount'],
        "approved_amount": meta['approved_amount'],
        "word_count": meta['word_count'],
        "transcript": entry['transcript'],
    }

    with open(json_path, 'w') as f:
        json.dump(sidecar, f, indent=2)

# Save combined metadata file
combined_path = os.path.join(OUTPUT_DIR, "call_metadata.json")
combined = {
    "generated_at": datetime.now().isoformat(),
    "total_calls": len(all_transcripts),
    "source_table": "salama_insurance.salama_silver.fact_claim",
    "calls": [
        {
            "call_id": e['metadata']['call_id'],
            "claim_id": e['metadata']['claim_id'],
            "scenario_type": e['metadata']['scenario_type'],
            "sentiment": e['metadata']['sentiment'],
            "business_line": e['metadata']['business_line'],
            "claim_status": e['metadata']['claim_status'],
            "duration_seconds": e['metadata']['estimated_duration_seconds'],
            "transcript": e['transcript'],
        }
        for e in all_transcripts
    ],
}

with open(combined_path, 'w') as f:
    json.dump(combined, f, indent=2)

print(f"✅ Individual JSON sidecar files saved ({len(all_transcripts)} files)")
print(f"✅ Combined metadata saved: {combined_path}")
print(f"   Total calls: {combined['total_calls']}")

In [0]:
import pandas as pd

# List all generated files
print("\n" + "="*70)
print("📁 GENERATED FILES")
print("="*70)

files = []
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fp)
    files.append({
        "File": f,
        "Type": "MP3 Audio" if f.endswith('.mp3') else "JSON Metadata" if f.endswith('.json') else "Other",
        "Size (KB)": round(size / 1024, 1),
    })

df_files = pd.DataFrame(files)
display(df_files)

# Summary stats
mp3_files  = [f for f in files if f['Type'] == 'MP3 Audio']
json_files = [f for f in files if f['Type'] == 'JSON Metadata']
total_mp3_kb  = sum(f['Size (KB)'] for f in mp3_files)
total_json_kb = sum(f['Size (KB)'] for f in json_files)

print(f"\n🎵 Audio files:    {len(mp3_files)} MP3s ({total_mp3_kb:.1f} KB total)")
print(f"📝 Metadata files: {len(json_files)} JSONs ({total_json_kb:.1f} KB total)")
print(f"📂 Output folder:  {OUTPUT_DIR}")

# Scenario distribution
print("\n" + "="*70)
print("🎭 SCENARIO DISTRIBUTION")
print("="*70)
display(df_results.groupby(['scenario', 'sentiment']).agg(
    calls=('claim_id', 'count'),
    avg_words=('words', 'mean'),
    avg_duration=('est_duration_sec', 'mean')
).reset_index())

# --- Copy instructions ---
print("\n" + "="*70)
print("🚀 NEXT STEPS: USE THESE FILES FOR CALL ANALYTICS")
print("="*70)
print("""
1. SPEECH-TO-TEXT:  Use Azure Speech Services or Whisper to transcribe
                    the MP3 files and compare against the JSON transcripts.

2. SENTIMENT:       Use the known 'sentiment' labels in metadata to train
                    or validate sentiment analysis models.

3. COMPLIANCE:      Check transcripts for required phrases (greeting,
                    claim number verification, closing).

4. COPY TO VOLUME:  To persist files in Unity Catalog:
   dbutils.fs.cp("file:/tmp/call_audio_files/",
                 "dbfs:/Volumes/salama_insurance/salama_silver/call_audio/",
                 recurse=True)
""")

In [0]:
import shutil
from pathlib import Path

src_dir = "/tmp/call_audio_files"
volume_path = "/Volumes/salama_insurance/salama_silver/call_audio"

# Create volume directory if it doesn't exist
Path(volume_path).mkdir(parents=True, exist_ok=True)

# Copy each file
copied_count = 0
total_kb = 0
for fname in sorted(os.listdir(src_dir)):
    src_file = os.path.join(src_dir, fname)
    dst_file = os.path.join(volume_path, fname)
    if os.path.isfile(src_file):
        shutil.copy2(src_file, dst_file)
        size_kb = round(os.path.getsize(dst_file) / 1024, 1)
        total_kb += size_kb
        copied_count += 1
        print(f"   ✅ {fname:50s} {size_kb:>8.1f} KB")

print(f"\n{'='*70}")
print(f"✅ Copied {copied_count} files to {volume_path}")
print(f"   Total size: {total_kb:.1f} KB ({total_kb/1024:.1f} MB)")

## ℹ️ About This Data

**These are synthetic call recordings** generated from real claims data in `salama_insurance.salama_silver.fact_claim`. The audio is produced using Google Text-to-Speech (gTTS) and represents simulated customer-agent interactions.

### Methodology
- **10 claims** randomly sampled across all statuses, business lines, and risk ratings
- **6 scenario types** mapped from `CLAIM_STATUS`: Reported, Under Investigation, Approved/Settled, Rejected/Closed, General Inquiry
- **Real claim data** (amounts, policy IDs, claim numbers, business lines) woven into natural dialogue
- **Metadata JSON** accompanies each audio file with ground truth labels for analytics validation

### Suggested Call Analytics Pipeline
1. **Ingest** → Copy MP3 + JSON files to a Unity Catalog Volume
2. **Transcribe** → Run Speech-to-Text (Azure Cognitive Services / OpenAI Whisper)
3. **Analyze** → Sentiment analysis, keyword extraction, compliance scoring
4. **Dashboard** → Build call analytics dashboard with Databricks AI/BI
5. **ML** → Train models on labeled call data (scenario, sentiment, claim outcome)

---
*Generated by Salama Insurance Claims Call Analytics Generator*